In [8]:
import PIL

import os
import base64
from io import BytesIO

import torch

import datasets
from transformers import Qwen3_5ForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

from utils import load_model, load_dataset


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cuda


In [9]:
ds_qa = load_dataset(cache_dir="../huggingface/", add_prefixes=True)

In [12]:
ds_qa["train"][0]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=CMYK size=309x272>,
 'question': 'use as few words as possible. do not reason about the problem. where are liver stem cells (oval cells) located?',
 'answer': 'in the canals of hering'}

In [ ]:
model, processor = load_model("Qwen/Qwen3.5-9B", peft=False, cache_dir="../huggingface")

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")


In [ ]:
def preprocess_dataset(example):
    img = example["image"]
    if img.mode not in ("RGB", "L"):
        img = img.convert("RGB")

    return {
        "prompt": [{"role": "user", "content": [
            {"type": "text", "text": example["question"]},
            {"type": "image"},
        ]}],
        "completion": [{"role": "assistant", "content": example["answer"]}],
        "images": [example["image"]]
    }

ds_qa_train = ds_qa["train"].map(
    preprocess_dataset,
    remove_columns=ds_qa["train"].column_names,
    num_proc=os.cpu_count(),
    writer_batch_size=500,
)

In [ ]:
def collate_fn(examples):
    texts, images = [], []

    for example in examples:
        messages = example["prompt"] + example["completion"]  # full conversation
        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
        images.append(example["images"][0])

    batch = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )
    batch["labels"] = batch["input_ids"].clone()
    return batch

In [ ]:
SYSTEM_PROMPT_ENTITY_EXTRACTION = """
As a knowledge analyzer, your task is to dissect and understand an image provided by the user. You are required to perform the following steps:
1. Summarize the image: Provide a concise summary of the entire image, capturing the main points and themes.
2. Extract Entities: Identify and list all significant "nouns" or entities present within the image. These entities should include but not limited to:
* Context: How the image seems to have been generated, such as by a tomography, microscopic imagery, scan of a medical report...
* Objects: All objects present in the scene, including organs, human tissues, cells and annotation such as arrows.
* Concepts: Any significant abstract ideas or themes that are central to the image.
Try to exhaust as many entities as possible. Your response should be structured in a JSON format to organize the information effectively.
Ensure that the summary is brief yet comprehensive, and the list of entities is detailed and accurate.
Here is the format you should use for your response:
{
"summary": "<A concise summary of the article>",
"entities": ["entity1", "entity2", ...]
}
"""

In [ ]:
ds_qa["train"][0]

In [ ]:
img = ds_qa["train"]["image"][0]
display(img)

In [ ]:
img = ds_qa["train"]["image"][500]
display(img)


messages = [
    {
        "role": "system",
        "content": [
            {"type": "text", "text": "You are a helpful assistant."}
        ]
    },
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": img,
            },
            # {"type": "text", "text": ds["train"][0]["question"]},
            {"type": "text", "text": SYSTEM_PROMPT_ENTITY_EXTRACTION},
        ],
    }
]
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}


generated_ids = model.generate(**inputs, max_new_tokens=2048)
output_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
print(output_text)